# Problem Book P1, Chapter 6: Random Integers (Python companion)

Part A checks the numbers of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. Only `numpy`, `sympy`, `scipy` and `matplotlib` are needed.

*Before running a cell, predict its output.*

In [ ]:
import math, random, itertools
from math import gcd, log, pi, sqrt
from fractions import Fraction as Fr
from collections import Counter
import numpy as np
from sympy import primerange, totient, factorint
from scipy.special import zeta, polygamma
import matplotlib.pyplot as plt
random.seed(6)

def squarefree_sieve(N):                 # sf[n] is True when n is squarefree, for 0 <= n <= N
    sf = np.ones(N + 1, dtype=bool); sf[0] = False
    for p in primerange(2, int(N**0.5) + 1):
        sf[p * p::p * p] = False
    return sf
def euler_product(local, P=10**6):       # prod over p < P of local(p)
    return math.prod(local(p) for p in primerange(2, P))

## Part A: checks

**Problem 6.1 (no uniform integer).** The set $E\cap B$ of (c): the even numbers of the blocks $[4^j,2\cdot4^j[$. Its proportion in
$[1,N]$ should swing between about $\frac13$ (at $N=2\cdot4^j$) and $\frac16$ (at $N=4^{j+1}$).

In [ ]:
def in_blocks(n):
    j = (n.bit_length() - 1) // 2          # 4^j <= n < 4^(j+1)
    return n < 2 * 4**j
EB = np.cumsum([0] + [1 if n % 2 == 0 and in_blocks(n) else 0 for n in range(1, 4**10 + 1)])
for j in range(5, 10):
    print(j, round(EB[2 * 4**j] / (2 * 4**j), 4), round(EB[4**(j + 1)] / 4**(j + 1), 4))

**Problem 6.2 (valuations).** The law of $v_2(n)$, and the paradox of (c): $\mathbb E[2^{v_2(n)}]$ grows like $\frac12\log_2N$
although three quarters of the integers have $2^{v_2(n)}\le2$.

In [ ]:
def v(n, p):
    k = 0
    while n % p == 0: n //= p; k += 1
    return k
N = 2**20
vals = [v(n, 2) for n in range(1, N + 1)]
c = Counter(vals)
print([round(c[k] / N, 5) for k in range(5)], [0.5**(k + 1) for k in range(5)])
print("mean v_2:", sum(vals) / N, " mean v_3:", sum(v(n, 3) for n in range(1, 3**12 + 1)) / 3**12, "(1/(p-1) = 1 and 0.5)")
for e in (12, 16, 20):
    M = 2**e
    print(e, sum(2**vals[n - 1] for n in range(1, M + 1)) / M, e / 2, " P(2^v <= 2):", sum(1 for n in range(1, M + 1) if vals[n - 1] <= 1) / M)

**Problem 6.3 (squarefree neighbours).** The two Euler products, the empirical densities up to $10^7$, the first run of four
non-squarefree integers (claimed at $242$), and a check that four consecutive squarefree integers never occur.

In [ ]:
N = 10**7
sf = squarefree_sieve(N + 3)
two = euler_product(lambda p: 1 - 2 / p**2); three = euler_product(lambda p: 1 - 3 / p**2)
print("pairs:", two, (sf[1:N + 1] & sf[2:N + 2]).mean())
print("triples:", three, (sf[1:N + 1] & sf[2:N + 2] & sf[3:N + 3]).mean())
print("four squarefree in a row:", (sf[1:N + 1] & sf[2:N + 2] & sf[3:N + 3] & sf[4:N + 4]).any())
def first_run(L):
    run = 0
    for n in range(1, N):
        run = run + 1 if not sf[n] else 0
        if run == L: return n - L + 1
print({L: first_run(L) for L in (2, 3, 4, 5)}, factorint(242), factorint(243), factorint(244), factorint(245))

**Problem 6.4 (pairwise coprime).** The Euler product for $k=3,4$ against a direct count over $\{1,\dots,60\}^3$ and a
Monte Carlo for $k=4$, and the independent guess $(6/\pi^2)^3$.

In [ ]:
def pairwise_prob(k): return euler_product(lambda p: (1 - 1 / p)**(k - 1) * (1 + (k - 1) / p))
n = 60
cnt = sum(1 for a in range(1, n + 1) for b in range(1, n + 1) if gcd(a, b) == 1
          for c in range(1, n + 1) if gcd(a, c) == 1 and gcd(b, c) == 1)
print("k=3:", pairwise_prob(3), cnt / n**3, " independent:", (6 / pi**2)**3)
rng = random.Random(1); T = 200000
mc = sum(1 for _ in range(T) if all(gcd(x, y) == 1 for x, y in itertools.combinations([rng.randrange(1, 10**9) for _ in range(4)], 2)))
print("k=4:", pairwise_prob(4), mc / T)
p = 2   # the prime-by-prime comparison of (b): local factor versus the independent guess
for p in (2, 3, 5, 7):
    print(p, (1 - 1 / p)**2 * (1 + 2 / p), (1 - 1 / p**2)**3)

**Problem 6.5 (a random fraction).** The parity patterns of $a'/b'$, and $P(3\mid b')=\frac14$.

In [ ]:
n = 1000
pat, d3 = Counter(), 0
for a in range(1, n + 1):
    for b in range(1, n + 1):
        g = gcd(a, b); a1, b1 = a // g, b // g
        pat[(a1 % 2, b1 % 2)] += 1; d3 += (b1 % 3 == 0)
print({k: round(v / n**2, 4) for k, v in pat.items()}, " P(3 | b') =", d3 / n**2)

**Problem 6.6 (squarefree $n^2+1$).** The product over $p\equiv1\pmod4$ and the empirical density up to $10^6$.

In [ ]:
dens = math.prod(1 - 2 / p**2 for p in primerange(2, 10**6) if p % 4 == 1)
M = 2 * 10**5
from sympy.ntheory import sqrt_mod
bad = np.zeros(M + 1, dtype=bool)
for p in primerange(5, M + 1):        # p^2 | n^2+1 needs p = 1 mod 4 and p^2 <= M^2+1
    if p % 4 != 1: continue
    for r in sqrt_mod(-1, p, all_roots=True):
        r2 = (r - (r * r + 1) * pow(2 * r, -1, p * p)) % (p * p)      # Hensel lift to a root modulo p^2
        bad[r2::p * p] = True
print(dens, 1 - bad[1:].mean())

**Problem 6.7 (two squarefree summands).** The constants of the counting argument, and a direct check below $14000$.

In [ ]:
P13 = math.prod(1 - 2 / p**2 for p in primerange(2, 14))
tail = sum(1 / p**2 for p in primerange(17, 10**7)) + 1 / (10**7 * log(10**7))   # tail beyond 10^7 is < 1/(x log x)
print("prod_{p<=13}(1-2/p^2) =", P13, " sum_{p>=17} 1/p^2 =", tail)
print("threshold:", 4097 / (0.3335 - 0.0331), " (positive beyond it)")
sf = squarefree_sieve(14000)
least = [next(a for a in range(1, n) if sf[a] and sf[n - a]) for n in range(2, 14000)]
print("all n < 14000 represented; largest least summand:", max(least))

**Problem 6.8 ($\varphi(n)/n$).** The first two moments against their Euler products, and the proportion of $n$ with
$\varphi(n)\le n/2$.

In [ ]:
N = 10**6
phi = np.arange(N + 1, dtype=float)
for p in primerange(2, N + 1):
    phi[p::p] *= (1 - 1 / p)
r = phi[1:] / np.arange(1, N + 1)
for k in (1, 2, 3):
    print(k, (r**k).mean(), euler_product(lambda p: 1 - 1 / p + (1 - 1 / p)**k / p))
print("6/pi^2 =", 6 / pi**2, " proportion with phi(n) <= n/2:", (r <= 0.5).mean())
xs = np.sort(r)
plt.figure(figsize=(6, 3.5)); plt.plot(xs, np.arange(1, N + 1) / N, lw=1)
plt.xlabel("x"); plt.ylabel("proportion of n with phi(n)/n <= x"); plt.title("A continuous, purely singular law"); plt.show()

**Problem 6.9 (the law of the gcd).** $P(\gcd=g)$, $\mathbb E\gcd-\frac6{\pi^2}\log N$, $\mathbb E\log\gcd$ and $\mathbb E\operatorname{lcm}/N^2$,
computed exactly with the count $\#\{(a,b):\gcd=g\}=\sum_{d}\mu(d)\lfloor N/gd\rfloor^2$.

In [ ]:
from sympy import mobius
N = 3000
mu = [0] + [int(mobius(d)) for d in range(1, N + 1)]
cop = lambda M: sum(mu[d] * (M // d)**2 for d in range(1, M + 1) if mu[d])     # pairs in [1,M]^2 with gcd 1
cnt = {g: cop(N // g) for g in range(1, N + 1)}
print([round(cnt[g] / N**2 * g * g * pi**2 / 6, 3) for g in range(1, 6)])
print("E gcd - 6/pi^2 log N:", sum(g * c for g, c in cnt.items()) / N**2 - 6 / pi**2 * log(N))
print("E log gcd:", sum(log(g) * c for g, c in cnt.items()) / N**2, " -zeta'(2)/zeta(2) = 0.569961")
n = 600
print("E lcm / N^2:", sum(a * b // gcd(a, b) for a in range(1, n + 1) for b in range(1, n + 1)) / n**4, zeta(3) / (4 * zeta(2)))

**Problem 6.10 (Fibonacci).** $\gcd(F_a,F_b)=F_{\gcd(a,b)}$ and the limit $\frac{15}{2\pi^2}$.

In [ ]:
F = [0, 1]
for _ in range(400): F.append(F[-1] + F[-2])
print(all(gcd(F[a], F[b]) == F[gcd(a, b)] for a in range(1, 120) for b in range(1, 120)))
N = 400
print(sum(1 for a in range(1, N + 1) for b in range(1, N + 1) if gcd(F[a], F[b]) == 1) / N**2, 15 / (2 * pi**2))
print("E log gcd(F_a,F_b) / log N:", sum(log(F[gcd(a, b)]) for a in range(1, N + 1) for b in range(1, N + 1)) / N**2 / log(N),
      " 6 log(phi)/pi^2 =", 6 * log((1 + sqrt(5)) / 2) / pi**2, "(the O(1) is still visible)")

**Problem 6.11 (visible points).** $V(R)$ against $\frac6\pi R^2$, and the share of visible points in the sector $0\le\arg<\frac\pi4$.

In [ ]:
for R in (50, 200, 800):
    V = sec = 0
    for a in range(-R, R + 1):
        for b in range(-R, R + 1):
            if a * a + b * b <= R * R and gcd(abs(a), abs(b)) == 1:
                V += 1; sec += (0 <= b < a)
    print(R, V, round(6 / pi * R * R), round((V - 6 / pi * R * R) / (R * log(R)), 3), " sector share:", round(sec / V, 4))

**Problem 6.12 (Pólya's orchard).** For small $R$, search the directions for the largest trunk radius that still leaves a
line of sight. The chapter proves that this critical radius lies in $[\frac1{\sqrt{R^2+1}},\frac1R[$.

In [ ]:
def critical(R, K=200000):
    pts = np.array([(a, b) for a in range(-R, R + 1) for b in range(-R, R + 1) if 0 < a * a + b * b <= R * R], float)
    best = 0
    th = np.linspace(0, pi / 4, K)        # by symmetry one octant is enough
    u = np.stack([np.cos(th), np.sin(th)], 1)
    for chunk in np.array_split(np.arange(K), 20):
        proj = pts @ u[chunk].T                                       # along the ray
        dist = np.abs(pts[:, :1] * u[chunk, 1] - pts[:, 1:] * u[chunk, 0])
        dist = np.where(proj > 0, dist, np.inf)
        best = max(best, dist.min(0).max())
    return best
for R in (2, 3, 4, 6, 8):
    print(R, round(1 / sqrt(R * R + 1), 5), round(critical(R), 5), round(1 / R, 5))

**Problem 6.13 (Farey).** $|\mathcal F_N|$, the determinant and denominator properties, $\sum\frac1{kk'}=1$ and the extreme gaps.

In [ ]:
def farey(N): return sorted({Fr(h, k) for k in range(1, N + 1) for h in range(k + 1)})
for N in (5, 30, 100):
    Fs = farey(N); pairs = list(zip(Fs, Fs[1:]))
    ok = all(y.numerator * x.denominator - x.numerator * y.denominator == 1 and x.denominator + y.denominator > N for x, y in pairs)
    gaps = [y - x for x, y in pairs]
    print(N, len(Fs), 1 + sum(int(totient(k)) for k in range(1, N + 1)), round(3 / pi**2 * N * N), ok,
          sum(Fr(1, x.denominator * y.denominator) for x, y in pairs), min(gaps) == Fr(1, N * (N - 1)), max(gaps) == Fr(1, N))

**Problem 6.14 (Euclid's game).** Brute force on all positions with $a<b\le150$, against the rule "the mover wins iff $b/a>\varphi$",
and the proportion of winning starts.

In [ ]:
from functools import lru_cache
import sys; sys.setrecursionlimit(10000)
@lru_cache(None)
def wins(a, b):                       # a <= b, player to move
    if b % a == 0: return True
    return any(not wins(*sorted((a, b - q * a))) for q in range(1, b // a + 1))
phi_ = (1 + sqrt(5)) / 2
M = 150
print(all(wins(a, b) == (b / a > phi_) for b in range(2, M + 1) for a in range(1, b)))
M = 2000
print(sum(1 for b in range(2, M + 1) for a in range(1, b) if b > phi_ * a) / (M * (M - 1) / 2), 1 / phi_)

**Problem 6.15 (the Gauss measure).** Invariance $\mu(T^{-1}[0,t[)=\mu([0,t[)$, and $P(a_1=k)$ under the uniform law and under $\mu$.

In [ ]:
mu_ = lambda a, b: (math.log1p(b) - math.log1p(a)) / log(2)
for t in (0.1, 0.5, 0.9):
    print(t, mu_(0, t), sum(mu_(1 / (k + t), 1 / k) for k in range(1, 10**6)))
print([(k, round(1 / k - 1 / (k + 1), 4), round(math.log2((k + 1)**2 / (k * (k + 2))), 4)) for k in (1, 2, 3, 10)])

**Problem 6.16 (Gauss–Kuzmin).** The constant $c$, the densities $f_n$ computed with the transfer operator on a grid, and the bound
$|f_n-g|\le c^n/(1+t)$ of (d).

In [ ]:
c1 = 0.5 + 2 * sum(1 / (k**3 * (k + 1)) for k in range(2, 10**6)) + sum((k - 1) / (k * (k + 1)**2) for k in range(2, 10**6))
print("c =", c1, 2 * zeta(3) - 1.5)
ts = np.linspace(0, 1, 401); K = 3000
ks = np.arange(1, K + 1)
def transfer(f):
    g = np.empty_like(ts)
    for i, t in enumerate(ts):
        u = 1 / (ks + t)
        g[i] = np.sum(np.interp(u, ts, f) * u * u) + f[0] / (K + t)    # tail: f(u) is about f(0) for u < 1/K
    return g
g = 1 / ((1 + ts) * log(2))
f = np.ones_like(ts); errs = []
for n in range(1, 7):
    f = transfer(f) if n > 1 else polygamma(1, 1 + ts)
    e = np.max(np.abs(f - g) * (1 + ts)); errs.append(e)
    print(n, round(e, 7), " bound c^n =", round(c1**n, 4))
print("successive ratios:", [round(errs[i + 1] / errs[i], 3) for i in range(len(errs) - 1)], "(Wirsing: 0.3037; see Lab L6.2)")

**Problem 6.17 (Lévy).** The integral of (b), and $\mathbb E[\log q_n]/n$ for random $x$ with $600$ binary digits.

In [ ]:
from scipy.integrate import quad
print(quad(lambda t: -log(t) / (1 + t), 0, 1)[0], pi**2 / 12)
def logq(m, B, n):                    # log of the convergent denominators of m / 2^B, up to index n
    a, b = 2**B, m; q0, q1 = 0, 1; out = []
    for _ in range(n):
        if a == 0 or b == 0: break
        k, r = divmod(a, b); a, b = b, r
        q0, q1 = q1, k * q1 + q0; out.append(math.log(q1))
    return out
rng = random.Random(3); n = 80
L = np.array([logq(rng.getrandbits(600) | 1, 600, n) for _ in range(3000)])
print("E log q_n at n = 20, 40, 80:", [round(L[:, k - 1].mean(), 2) for k in (20, 40, 80)])
print("slope:", (L[:, 79].mean() - L[:, 39].mean()) / 40, pi**2 / (12 * log(2)))

**Problem 6.18 (Benford).** The proportion of leading $1$'s at $10^k-1$ and $2\cdot10^k-1$, and the logarithmic density.

In [ ]:
N = 2 * 10**6
lead1 = np.cumsum([0] + [1 if str(n)[0] == "1" else 0 for n in range(1, N + 1)])
print([(round(lead1[10**k - 1] / (10**k - 1), 4), round(lead1[2 * 10**k - 1] / (2 * 10**k - 1), 4)) for k in range(2, 7)])
for d in (1, 2, 9):
    s = sum(1 / n for n in range(1, N + 1) if str(n)[0] == str(d))
    print(d, round(s / log(N), 4), round(math.log10(1 + 1 / d), 4), "(convergence is slow, with oscillations)")

**Problem 6.19 (binary digits).** The law of $s(n)$ for $N=2^{16}$, and the normalised law for $N=10^6$, which is not a power of $2$.

In [ ]:
k = 16
c = Counter(bin(n).count("1") for n in range(2**k))
print(all(c[j] == math.comb(k, j) for j in range(k + 1)))
N = 10**6; m = math.log2(N)
z = np.array([(bin(n).count("1") - m / 2) / sqrt(m / 4) for n in range(N)])
print("mean, variance, P(Z <= 1):", round(z.mean(), 3), round(z.var(), 3), round((z <= 1).mean(), 3), " Gaussian: 0, 1, 0.841")

**Problem 6.20 (squares modulo $p^k$).** The formula for $S(p^k)$, the limits $\frac p{2(p+1)}$ and $\frac16$, and multiplicativity.

In [ ]:
S = lambda m: len({x * x % m for x in range(m)})
print(all(S(p**k) == 1 + sum((p - 1) // 2 * p**(k - 2 * j - 1) for j in range(0, (k + 1) // 2))
          for p in (3, 5, 7) for k in range(1, 6)))
print([(p, round(S(p**5) / p**5, 5), round(p / (2 * (p + 1)), 5)) for p in (3, 5, 7)])
print([round(S(2**k) / 2**k, 5) for k in (8, 12, 16)], 1 / 6)
print(all(S(a * b) == S(a) * S(b) for a in range(1, 40) for b in range(1, 40) if gcd(a, b) == 1))
m = math.prod(primerange(2, 30))
print("proportion of squares modulo the primorial of 29:", math.prod(S(p) / p for p in primerange(2, 30)))

**Problem 6.21 (universal hashing).** Exhaustive over all $(a,b)$ for $p=101$, $m=7$: the worst collision probability against
$\frac1m+\frac1p$.

In [ ]:
p, m = 101, 7
worst = 0
for x in range(p):
    for y in range(x + 1, p):
        col = sum(1 for a in range(p) for b in range(p) if (a * x + b) % p % m == (a * y + b) % p % m)
        worst = max(worst, col / p**2)
print(worst, 1 / m + 1 / p)

**Problem 6.22 (Chung–Diaconis–Graham).** The exact law of $X_n$ for $p=1009$, its total variation distance $d_n$ to uniform, the
lower bound $1-3^n/p$ and the upper bound $\frac12\sqrt p\,3^{-\lfloor n/L\rfloor}$.

In [ ]:
def cdg_tv(p, nmax):
    dist = np.zeros(p); dist[0] = 1; out = []
    idx = (2 * np.arange(p)) % p
    for n in range(1, nmax + 1):
        new = np.zeros(p)
        for e in (-1, 0, 1):
            np.add.at(new, (idx + e) % p, dist / 3)
        dist = new; out.append(0.5 * np.abs(dist - 1 / p).sum())
    return out
p = 1009; L = math.ceil(math.log2(p)) + 1
d = cdg_tv(p, 60)
for n in (3, 6, 9, 12, 20, 30, 45, 60):
    print(n, round(d[n - 1], 5), " lower:", round(max(0, 1 - 3**n / p), 4), " upper:", round(min(1, 0.5 * sqrt(p) * 3.0**(-(n // L))), 4))

## Part B: laboratory

**L6.1 (visible points in space).** The proportion of lattice points of the cube $[-R,R]^3$ with $\gcd(a,b,c)=1$ should tend to
$1/\zeta(3)$. Check it, and find the analogue of Problem 6.11(b) for the ball.

**L6.2 (Wirsing's constant).** The ratios of successive errors in Problem 6.16 approach the second eigenvalue $-0.3036630\ldots$ of
the transfer operator. Estimate it by power iteration on a discretisation, after removing the Gauss density.

**L6.3 (Khinchin's constant).** For almost every $x$, $(a_1\cdots a_n)^{1/n}\to K_0=2.6854\ldots$ Check it for random $x$ (exact
arithmetic), and explain from Problem 6.15 why $\log K_0=\sum_k\log_2\frac{(k+1)^2}{k(k+2)}\log k$.

**L6.4 (why doubling helps).** Compare the walk of Problem 6.22 with the plain walk $X_{n+1}=X_n+\varepsilon_n$: how many steps
does each need to bring $d_n$ below $\frac14$ for $p=101,211,401$? Fit the growth.

*Try them before reading on.*

### Solutions
### L6.1

In [ ]:
for R in (20, 40, 80):
    r = np.arange(-R, R + 1)
    A, B, Cc = np.meshgrid(r, r, r, indexing="ij")
    g = np.gcd(np.gcd(A, B), Cc)
    print(R, round((g == 1).sum() / ((2 * R + 1)**3 - 1), 5), round(1 / zeta(3), 5),
          " ball:", round(((g == 1) & (A**2 + B**2 + Cc**2 <= R * R)).sum() / (4 / 3 * pi * R**3), 5))

Both proportions approach $1/\zeta(3)=0.8319$: a point is hidden exactly when some prime divides all three coordinates, which
happens with probability $p^{-3}$ for each $p$. The analogue of 6.11(b) is $V_3(R)=\frac{4\pi}{3\zeta(3)}R^3+O(R^2)$, with the
same Möbius proof; in dimension $3$ the error no longer needs the $\log$.

### L6.2

In [ ]:
ts = np.linspace(0, 1, 801); K = 3000; ks = np.arange(1, K + 1)
def transfer(f):
    g = np.empty_like(ts)
    for i, t in enumerate(ts):
        u = 1 / (ks + t)
        g[i] = np.sum(np.interp(u, ts, f) * u * u) + f[0] / (K + t)
    return g
gd = 1 / ((1 + ts) * log(2))
h = np.ones_like(ts) - gd                     # integral 0: no component along the Gauss density
for n in range(12):
    h2 = transfer(h)
    lam = np.dot(h2, h) / np.dot(h, h)
    h = h2 / np.max(np.abs(h2))
print("second eigenvalue estimate:", round(lam, 5), " Wirsing: -0.3036630")

The operator preserves integrals, so $h=f_0-g$ with integral $0$ stays orthogonal to the top eigenvalue $1$, and power iteration
finds the next one. It is negative: the error changes sign at each step, which is visible in the figure, where $f_1$ and $f_2$
lie on opposite sides of the dashed curve.

### L6.3

In [ ]:
K0 = math.exp(sum(math.log2((k + 1)**2 / (k * (k + 2))) * log(k) for k in range(1, 10**6)))
rng = random.Random(7)
def quotients(m, B, n):
    a, b = 2**B, m; out = []
    while b and len(out) < n:
        k, r = divmod(a, b); a, b = b, r; out.append(k)
    return out
gm = [math.exp(np.mean(np.log(quotients(rng.getrandbits(4000) | 1, 4000, 2000)[1:]))) for _ in range(20)]
print("K0 from the series:", round(K0, 4), " geometric means for 20 random x:", round(np.mean(gm), 4), round(np.std(gm), 4))

Birkhoff's ergodic theorem for the Gauss map (Part II) says that $\frac1n\sum_{j<n}F(T^jx)\to\int F\,d\mu$ for almost every $x$.
With $F(x)=\log a_1(x)$ the right side is $\sum_k\mu(a_1=k)\log k$, which is the series. The first quotient is dropped because
$m/2^B$ starts with $a_0=\lfloor 2^B/m\rfloor$ in this code. Individual geometric means still fluctuate by a few percent at
$n=2000$: the convergence is slow because $\log a$ has a heavy tail.

### L6.4

In [ ]:
def plain_tv_steps(p):
    dist = np.zeros(p); dist[0] = 1; n = 0
    while True:
        dist = (np.roll(dist, 1) + dist + np.roll(dist, -1)) / 3; n += 1
        if 0.5 * np.abs(dist - 1 / p).sum() < 0.25: return n
def cdg_steps(p):
    d = cdg_tv(p, 200)
    return next(n for n, x in enumerate(d, 1) if x < 0.25)
rows = [(p, plain_tv_steps(p), cdg_steps(p)) for p in (101, 211, 401)]
print(rows)
print("plain / p^2:", [round(a / p**2, 3) for p, a, b in rows], "  doubling / log(p):", [round(b / log(p), 2) for p, a, b in rows])

The plain walk needs a number of steps proportional to $p^2$ (its Fourier coefficients are $\frac{1+2\cos(2\pi t/p)}3$, and for
$t=1$ this is $1-O(p^{-2})$). The doubling walk needs a number of steps of order $\log p$ here, even better than the $\log^2p$ of
Problem 6.22(d). Chung, Diaconis and Graham showed that $C\log p\log\log p$ steps always suffice, that about $1.02\log_2p$
steps suffice for almost every $p$, and that for $p=2^t-1$ (where $2$ has small order) $c\log p\log\log p$ steps are needed.